# Automated Citation Adder for Research Introduction
## Pipeline: Extract Claims → Score Relevance → Optimize Assignment → Insert Citations

This notebook:
1. Parses PDFs from papers/ folder and builds ChromaDB vector store
2. Identifies factual claims in introduction.txt
3. Scores each paper's relevance to each claim using LLM
4. Runs simulated annealing to optimize citation assignment
5. Generates introduction_with_citations.txt and references.txt

In [1]:
# Import Required Libraries
import os
import glob
import time
import json
import re
import random
import numpy as np
from typing import List, Dict, Tuple, Set
import getpass
from datetime import datetime
from collections import defaultdict, Counter

# LLM and Document Processing
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader

# Vector Store
import chromadb
from chromadb.utils import embedding_functions

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# API Keys
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

print("✓ All imports successful")

/Users/christopherwaight/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ All imports successful


In [34]:
# Configuration Parameters
CONFIG = {
    # File paths
    'PAPERS_FOLDER': './papers',
    'INTRODUCTION_FILE': './introduction.txt',
    'OUTPUT_INTRO': './introduction_with_citations.txt',
    'OUTPUT_REFS': './references.txt',
    
    # Citation constraints
    'MIN_PAPERS': 8,
    'MAX_CITES_PER_PAPER': 5,
    'CITES_PER_CLAIM_MIN': 1,
    'CITES_PER_CLAIM_MAX': 3,
    'WEAK_MATCH_THRESHOLD': 0.15,
    
    # Vector similarity
    'TOP_K_CANDIDATES': 8,
    
    # Simulated annealing
    'SA_ITERATIONS': 2200,
    'SA_RUNS': 100,
    'SA_INITIAL_TEMP': 1.1,
    'SA_COOLING_RATE': 0.993,
    'SA_MIN_TEMP': 0.0001,
    
    # LLM settings
    'LLM_MODEL': 'gpt-4o',  # Most powerful general-purpose OpenAI model
    'LLM_TEMP_EXTRACTION': 0.1,
    'LLM_TEMP_SCORING': 0.0,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  PAPERS_FOLDER: ./papers
  INTRODUCTION_FILE: ./introduction.txt
  OUTPUT_INTRO: ./introduction_with_citations.txt
  OUTPUT_REFS: ./references.txt
  MIN_PAPERS: 8
  MAX_CITES_PER_PAPER: 5
  CITES_PER_CLAIM_MIN: 1
  CITES_PER_CLAIM_MAX: 3
  WEAK_MATCH_THRESHOLD: 0.15
  TOP_K_CANDIDATES: 8
  SA_ITERATIONS: 2200
  SA_RUNS: 100
  SA_INITIAL_TEMP: 1.1
  SA_COOLING_RATE: 0.993
  SA_MIN_TEMP: 0.0001
  LLM_MODEL: gpt-4o
  LLM_TEMP_EXTRACTION: 0.1
  LLM_TEMP_SCORING: 0.0


## Step 1: Load and Process PDFs

In [3]:
def load_pdfs_from_folder(folder_path):
    """
    Load all PDFs from folder and extract text content.
    Returns list of dicts with filename, content, and metadata.
    """
    pdf_paths = glob.glob(f"{folder_path}/*.pdf")
    papers = []
    
    print(f"\nFound {len(pdf_paths)} PDF files in {folder_path}")
    print("Loading PDFs...\n")
    
    for i, path in enumerate(pdf_paths, 1):
        filename = os.path.basename(path)
        print(f"  [{i}/{len(pdf_paths)}] {filename[:60]}...", end="")
        
        try:
            loader = PyPDFLoader(path)
            pages = loader.load()
            
            # Combine all pages (or first 20 for token management)
            content = ""
            for page_num, page in enumerate(pages[:20]):
                content += f" {page.page_content}"
            
            # Extract basic metadata
            first_page = pages[0].page_content if pages else ""
            lines = first_page.split('\n')[:15]
            title = lines[0].strip() if lines else filename.replace('.pdf', '')
            
            # Try to extract year
            year_match = re.search(r'(19|20)\d{2}', path + first_page)
            year = year_match.group() if year_match else "Unknown"
            
            # Try to extract authors from first page
            authors = "Unknown"
            for line in lines[1:8]:
                if re.search(r'[A-Z][a-z]+ [A-Z]', line):
                    authors = line.strip()[:100]
                    break
            
            papers.append({
                'id': i,
                'filename': filename,
                'title': title[:500],
                'authors': authors,
                'year': year,
                'content': content,
                'first_page': first_page[:4000]
            })
            
            print(" ✓")
            
        except Exception as e:
            print(f" ✗ Error: {e}")
    
    print(f"\n✓ Successfully loaded {len(papers)} papers")
    return papers

# Load papers
papers = load_pdfs_from_folder(CONFIG['PAPERS_FOLDER'])


Found 54 PDF files in ./papers
Loading PDFs...

  [1/54] [2] Citation Trakas P S Tantoulas A Bechlioulis CP Formation... ✓
  [2/54] [9] Distributed Multi-Robot Active-Sensing of a Diffusive So... ✓
  [3/54] [30] Multiple UAV Adaptive Navigation for Three-Dimensional ... ✓
  [4/54] [13] EntrapmentEscorting and Patrolling Missions - Missions....

Ignoring wrong pointing object 5 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)


 ✓
  [5/54] [52] Structured Isosurface Mapping of 3D Scalar Fields - Iso... ✓
  [6/54] [15] Finite-Time Estimation and Control for Multi-Aircraft S... ✓
  [7/54] [39] Precise Landing of Autonomous Aerial Vehicles Using Vec... ✓
  [8/54] [25] Initial Study of Multirobot Adaptive Navigation for - S... ✓
  [9/54] [50] Spiral Search Pattern for Scalable Assemblages of - Sea... ✓
  [10/54] [5] Connected and automated vehicles CA Vs and robot swarms ...

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 41 0 (offset 0)
Ignoring wrong pointing object 79 0 (offset 0)
Ignoring wrong pointing object 86 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 154 0 (offset 0)
Ignoring wrong pointing object 156 0 (offset 0)
Ignoring wrong pointing object 158 0 (offset 0)


 ✓
  [11/54] [19] A Functional Indoor Testbed for 2D Vector Field Multiro... ✓
  [12/54] [4] Co-Evolution of Multi-Robot Controllers and Task Cues fo...

...

 ✓
  [40/54] [53] Unifying Control Architecture for Reactive Particle Swa... ✓
  [41/54] [26] Circular Formation Control - 2018.pdf... ✓
  [42/54] [47] SICE Journal of Control Measurement and System Integrat... ✓
  [43/54] [11] Dynamic Elliptical Shaping Control for Swarm Robots - K... ✓
  [44/54] [29] Motion Planning and Collision Avoidance using Navigatio... ✓
  [45/54] [49] Sparsity Structure and Optimality of Multi-Robot Covera... ✓
  [46/54] [48] Singularity-Free Guiding Vector Field for Robot Navigat...

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)


 ✓
  [47/54] [40] Proceedings of the ASME 2019 International Design Engin... ✓
  [48/54] [12] Dynamic Plume T racking by Cooperative Robots Jun-Wei W... ✓
  [49/54] [24] ICA T An Indoor Connected and Autonomous T estbed for -... ✓
  [50/54] [41] Quaternions and Dual Quaternions Singularity-Free Multi...

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)


 ✓
  [51/54] [36] Obstacle Avoidance Using Complex Vector Fields - Avoida... ✓
  [52/54] [10] Distributed Optimal Formation for Uncertain Euler-Lagra... ✓
  [53/54] [23] Himangshu Kalita Ravi Teja Nallapu Andrew Warren and - ... ✓
  [54/54] [31] Multi-Robot Object SLAM Using Distributed Variational I... ✓

✓ Successfully loaded 54 papers


## Step 2: Build ChromaDB Vector Store

In [4]:
def build_vector_store(papers):
    """
    Build ChromaDB vector store from paper contents.
    Returns ChromaDB collection.
    """
    print("\nBuilding ChromaDB vector store...")
    
    # Initialize ChromaDB client
    client = chromadb.Client()
    
    # Create embedding function using OpenAI
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.environ.get("OPENAI_API_KEY"),
        model_name="text-embedding-3-small"
    )
    
    # Create or get collection
    collection = client.create_collection(
        name="papers_collection",
        embedding_function=openai_ef,
        metadata={"description": "Research papers for citation matching"}
    )
    
    # Add papers to collection
    documents = []
    metadatas = []
    ids = []
    
    for paper in papers:
        # Use title + first page + sample of content for embedding
        doc_text = f"{paper['title']}. {paper['first_page']} {paper['content'][:5000]}"
        documents.append(doc_text)
        
        metadatas.append({
            'filename': paper['filename'],
            'title': paper['title'][:100],
            'year': paper['year']
        })
        
        ids.append(f"paper_{paper['id']}")
    
    print(f"  Adding {len(documents)} documents to vector store...")
    
    # Add in batches to avoid rate limits
    batch_size = 30
    for i in range(0, len(documents), batch_size):
        batch_end = min(i + batch_size, len(documents))
        collection.add(
            documents=documents[i:batch_end],
            metadatas=metadatas[i:batch_end],
            ids=ids[i:batch_end]
        )
        print(f"    Added batch {i//batch_size + 1}/{(len(documents)-1)//batch_size + 1}")
        time.sleep(0.5)  # Rate limiting
    
    print("✓ Vector store built successfully\n")
    return collection

# Build vector store
collection = build_vector_store(papers)


Building ChromaDB vector store...
  Adding 54 documents to vector store...
    Added batch 1/2
    Added batch 2/2
✓ Vector store built successfully



## Step 3: Load Introduction and Identify Claims

In [5]:
def load_introduction(filepath):
    """Load introduction text from file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Remove line numbers if present (format: "1→text")
    content = re.sub(r'^\s*\d+→', '', content, flags=re.MULTILINE)
    
    return content.strip()

# Load introduction
introduction_text = load_introduction(CONFIG['INTRODUCTION_FILE'])
print(f"✓ Loaded introduction: {len(introduction_text)} characters\n")
print("First 300 characters:")
print(introduction_text[:300] + "...\n")

✓ Loaded introduction: 5402 characters

First 300 characters:
**Introduction**

Adaptive navigation of multirobot systems through physical environments has successfully been applied to search and rescue and environmental monitoring missions. In most cases, these environments are modeled as scalar fields representing quantities like temperature or chemical conc...



In [6]:
def extract_claims(introduction_text, llm):
    """
    Use LLM to identify factual claims that need citations.
    Returns list of claim dicts with text and position.
    """
    print("\nExtracting factual claims from introduction...")

    claim_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are identifying factual claims in a research introduction that need citations.

A claim is a factual statement that should be supported by prior work. EXTRACT these:
- Statements about what prior work has done/shown/demonstrated
- Technical capabilities or limitations of existing methods
- Problems or challenges identified by the research community
- Infrastructure capabilities (testbeds, sensors, tracking systems, hardware platforms)
- Validation challenges (sim-to-real gap, hardware limitations, sensor noise, communication delays)
- Mathematical or theoretical foundations mentioned (eigenvalue analysis, Jacobian estimation, etc.)
- State-of-the-art performance or characteristics
- Established methods, approaches, or control architectures
- Implementation challenges when transitioning to hardware

EXTRACT these specific types (even if they seem somewhat general):
✓ "Indoor experimental facilities enable validation" (testbed capabilities - cite testbed papers)
✓ "Motion capture systems provide ground truth localization" (sensor technology - cite tracking papers)
✓ "Gap persists between simulation and hardware" (validation challenges - cite sim-to-real papers)
✓ "Communication delays affect performance" (implementation issues)
✓ "Cluster control enables coordination" (control architectures - cite cluster control papers)
✓ "Eigenvalue analysis reveals critical point types" (mathematical methods - cite theory papers)
✓ "Formation control maintains spacing" (cite formation control papers)
✓ "Distributed estimation uses measurements from multiple robots" (cite distributed sensing papers)

DO NOT extract (negative style guide):
✗ Authors' own contributions: "This paper presents", "We develop", "Our approach uses"
✗ Authors' own results: "Our experimental validation demonstrates centimeter level accuracy"
✗ Authors' own specifications: "in a two meter square testbed operating at 10 Hz"
✗ Paper organization: "Section II describes", "The remainder of this paper", "Section III presents"

IMPORTANT: Break compound sentences into separate claims. If a sentence makes multiple distinct factual statements, extract each one separately.

Example:
Sentence: "Precision motion capture systems provide ground truth localization at high rates, enabling validation of theoretical predictions."
→ Extract TWO claims:
  1. "Precision motion capture systems provide ground truth localization at high rates"
  2. "Ground truth localization enables validation of theoretical predictions"

Extract each independent claim separately, even if multiple appear in one sentence."""),
        ("human", """Extract all factual claims from this introduction that need citations.

For each claim, provide:
1. The exact text of the claim (can be partial sentence)
2. A brief topic/keyword for the claim

Format your response as a JSON array:
[
  {{"claim": "exact text", "topic": "brief topic"}},
  ...
]

Introduction text:
{intro_text}
""")
    ])

    chain = claim_prompt | llm
    response = chain.invoke({"intro_text": introduction_text})

    # Parse JSON response
    try:
        # Extract JSON from response (handle markdown code blocks)
        content = response.content
        json_match = re.search(r'```json\s*([\s\S]*?)```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            # Try to find JSON array directly
            json_match = re.search(r'\[\s*{[\s\S]*}\s*\]', content)
            json_str = json_match.group(0) if json_match else content

        claims_data = json.loads(json_str)

        # Add IDs and positions
        claims = []
        for i, claim_data in enumerate(claims_data, 1):
            # Find position in text (approximate)
            claim_text = claim_data['claim']
            # Normalize for matching
            normalized_claim = re.sub(r'\s+', ' ', claim_text).strip()
            normalized_intro = re.sub(r'\s+', ' ', introduction_text)

            position = normalized_intro.find(normalized_claim[:50])  # Match first 50 chars

            claims.append({
                'id': i,
                'claim': claim_text,
                'topic': claim_data.get('topic', 'general'),
                'position': position if position != -1 else i * 100,  # Fallback ordering
                'citations': []  # Will be filled by optimization
            })

        # Sort by position to maintain order
        claims.sort(key=lambda x: x['position'])

        print(f"✓ Extracted {len(claims)} claims\n")
        return claims

    except json.JSONDecodeError as e:
        print(f"Error parsing LLM response: {e}")
        print(f"Response was: {response.content[:500]}")
        return []

# Initialize LLM for claim extraction
llm_extractor = ChatOpenAI(
    model=CONFIG['LLM_MODEL'],
    temperature=CONFIG['LLM_TEMP_EXTRACTION']
)

# Extract claims
claims = extract_claims(introduction_text, llm_extractor)

# Display first few claims
print("Sample claims:")
for claim in claims[:5]:
    print(f"  [{claim['id']}] ({claim['topic']}): {claim['claim'][:100]}...")



Extracting factual claims from introduction...
✓ Extracted 22 claims

Sample claims:
  [1] (applications of multirobot systems): Adaptive navigation of multirobot systems through physical environments has successfully been applie...
  [2] (scalar field modeling): Environments are modeled as scalar fields representing quantities like temperature or chemical conce...
  [3] (gradient methods in control): Control primitives based on gradient methods enable robots to find extrema, saddle points, ridges, o...
  [4] (effectiveness of scalar field approaches): Scalar field approaches have proven effective for many applications where the environment can be cha...
  [5] (limitations of scalar field representation): Many physical phenomena cannot be adequately represented as scalar fields...


## Step 4: Score Paper Relevance for Each Claim

In [7]:
def get_candidate_papers(claim_text, collection, top_k=10):
    """
    Use vector similarity to find top-k candidate papers for a claim.
    Returns list of paper IDs.
    """
    results = collection.query(
        query_texts=[claim_text],
        n_results=top_k
    )
    
    # Extract paper IDs from result IDs (format: "paper_N")
    paper_ids = []
    for doc_id in results['ids'][0]:
        paper_id = int(doc_id.split('_')[1])
        paper_ids.append(paper_id)
    
    return paper_ids

def extract_paper_sections(paper):
    """
    Extract key sections from paper for better scoring.
    Returns dict with abstract, intro, conclusion excerpts.
    """
    content = paper['content'].lower()
    first_page = paper['first_page']
    
    sections = {
        'title': paper['title'],
        'abstract': '',
        'introduction': '',
        'conclusion': '',
        'full_excerpt': first_page[:2000]
    }
    
    # Try to extract abstract (usually in first page)
    abstract_patterns = [
        r'abstract[\s\n]+(.*?)(?:introduction|keywords|1\.|i\.)',
        r'abstract[\s\n]+(.*?)(?:\n\n|\r\n\r\n)',
    ]
    for pattern in abstract_patterns:
        match = re.search(pattern, first_page.lower(), re.DOTALL | re.IGNORECASE)
        if match:
            sections['abstract'] = match.group(1)[:800].strip()
            break
    
    # Try to extract introduction
    intro_patterns = [
        r'(?:introduction|1\.[\s]*introduction)[\s\n]+(.*?)(?:2\.|ii\.|related work|background)',
        r'(?:introduction|1\.)[\s\n]+(.*?)$'
    ]
    for pattern in intro_patterns:
        match = re.search(pattern, content[:5000], re.DOTALL | re.IGNORECASE)
        if match:
            sections['introduction'] = match.group(1)[:1000].strip()
            break
    
    # Try to extract conclusion (usually at end)
    conclusion_patterns = [
        r'(?:conclusion|discussion|summary)[\s\n]+(.*?)(?:references|acknowledgment|$)',
    ]
    for pattern in conclusion_patterns:
        match = re.search(pattern, content[-3000:], re.DOTALL | re.IGNORECASE)
        if match:
            sections['conclusion'] = match.group(1)[:800].strip()
            break
    
    return sections

def score_paper_for_claim(claim_text, paper, llm):
    """
    Advanced scoring: multi-section analysis with reasoning and evidence extraction.
    Returns precise score between 0.00 and 1.00 on a continuous scale.
    """
    # Extract relevant sections
    sections = extract_paper_sections(paper)
    
    # Build context from multiple sections
    context_parts = []
    if sections['abstract']:
        context_parts.append(f"ABSTRACT: {sections['abstract']}")
    if sections['introduction']:
        context_parts.append(f"INTRODUCTION: {sections['introduction']}")
    if sections['conclusion']:
        context_parts.append(f"CONCLUSION: {sections['conclusion']}")
    
    if not context_parts:
        context_parts.append(f"EXCERPT: {sections['full_excerpt']}")
    
    context = "\n\n".join(context_parts)
    
    # Advanced scoring prompt with precise granular scoring (ESCAPED CURLY BRACES)
    scoring_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert research librarian evaluating citation relevance with PRECISE scoring.

Your task: Score how well this paper supports the claim on a CONTINUOUS 0.00-1.00 scale.

IMPORTANT: Use the FULL granularity of the scale. Don't round to 0.3, 0.5, 0.8. 
Use precise values like 0.73, 0.42, 0.88, 0.15, etc.

Scoring Guidelines (use precise values BETWEEN these ranges):

0.90-1.00: Perfect match
  - Paper's main contribution directly proves/demonstrates this exact claim
  - Contains experimental data, theorems, or comprehensive evidence
  - This claim could be the paper's title
  Examples: 0.95 (perfect + recent), 0.92 (perfect but older), 0.88 (very strong but minor gaps)

0.75-0.89: Strong direct evidence
  - Paper explicitly addresses this claim as a key point
  - Provides substantial evidence (data, analysis, implementation)
  - Claim is in introduction or conclusion as major finding
  Examples: 0.84 (strong with data), 0.79 (strong but less central), 0.76 (direct but brief)

0.60-0.74: Good supporting evidence
  - Paper addresses this claim clearly but not as main focus
  - Evidence exists but may be partial or in specific context
  - Mentioned multiple times but not deeply analyzed
  Examples: 0.68 (solid support), 0.65 (good but specific context), 0.61 (adequate evidence)

0.45-0.59: Moderate relevance
  - Paper discusses related concepts that support claim indirectly
  - Provides useful context or background for understanding claim
  - Methods/approaches are relevant even if exact claim not stated
  Examples: 0.54 (relevant methods), 0.49 (useful context), 0.46 (related but indirect)

0.30-0.44: Tangential connection
  - Paper mentions related topics in passing
  - Uses similar terminology but different focus
  - Could provide background context in related work section
  Examples: 0.38 (mentions topic), 0.34 (similar area), 0.31 (weak connection)

0.15-0.29: Minimal relevance
  - Very distant connection to claim
  - Same broad field but different specific topic
  - Keywords overlap but substance doesn't
  Examples: 0.22 (same field), 0.18 (keyword match only), 0.15 (very weak)

0.00-0.14: Not relevant
  - No meaningful connection to claim
  - Different research area entirely
  Examples: 0.08 (completely different), 0.03 (wrong field), 0.00 (no connection)

Evaluation Process:
1. Does the paper provide evidence FOR this specific claim?
2. How strong is the evidence? (data > theory > discussion > mention)
3. How central is this to the paper? (main point > key result > supporting > tangential)
4. What type of citation? (evidence > method > background > comparison)
5. Assign a PRECISE score based on these factors

Respond in JSON format (use this exact structure):
{{
  "reasoning": "2-3 sentence explanation of relevance and score rationale",
  "evidence_quote": "specific sentence from paper supporting claim, or empty string",
  "citation_type": "evidence|method|background|comparison|none",
  "score": 0.73,
  "confidence": "high|medium|low",
  "evidence_strength": "strong|moderate|weak|none"
}}

Note: The score value should be a precise decimal like 0.73, 0.42, 0.88, NOT rounded like 0.3, 0.5, 0.8"""),
        ("human", """Claim to support: "{claim}"

Paper title: {title}

Paper content:
{context}

Provide your evaluation with a PRECISE score (e.g., 0.67, 0.82, 0.43):"""),
    ])
    
    try:
        chain = scoring_prompt | llm
        response = chain.invoke({
            "claim": claim_text,
            "title": sections['title'],
            "context": context[:3500]  # Slightly more context for better precision
        })
        
        # Parse JSON response
        content = response.content.strip()
        
        # Extract JSON from response (handle markdown code blocks)
        json_match = re.search(r'```json\s*([\s\S]*?)```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            # Try to find JSON object directly
            json_match = re.search(r'\{[\s\S]*\}', content)
            json_str = json_match.group(0) if json_match else content
        
        result = json.loads(json_str)
        
        # Extract and validate score
        score = float(result.get('score', 0.0))
        
        # Round to 2 decimal places for consistency
        score = round(score, 2)
        score = max(0.0, min(1.0, score))  # Clamp to [0, 1]
        
        # Apply confidence penalty (reduce score if low confidence)
        confidence = result.get('confidence', 'medium').lower()
        evidence_strength = result.get('evidence_strength', 'moderate').lower()
        
        if confidence == 'low':
            score *= 0.85  # Reduce by 15% if low confidence
        
        # Additional calibration: boost high-quality matches, penalize weak ones
        if evidence_strength == 'strong' and score > 0.60:
            score = min(1.0, score * 1.05)  # Slight boost for strong evidence
        elif evidence_strength == 'weak' and score > 0.30:
            score *= 0.90  # Reduce weak evidence scores
        
        # Final rounding after adjustments
        score = round(score, 2)
        
        return score
        
    except json.JSONDecodeError as e:
        # Fallback: try to extract just the score
        print(f"    JSON parse error for paper {paper['id']}, trying fallback...")
        try:
            score_match = re.search(r'"score":\s*(\d+\.?\d*)', response.content)
            if score_match:
                score = float(score_match.group(1))
                return round(max(0.0, min(1.0, score)), 2)
        except:
            pass
        print(f"    Could not parse response: {response.content[:200]}")
        return 0.0
        
    except Exception as e:
        print(f"    Error scoring paper {paper['id']}: {e}")
        return 0.0

def build_relevance_matrix(claims, papers, collection):
    """
    Build matrix of relevance scores: scores[claim_id][paper_id] = score.
    Also returns candidate_papers[claim_id] = list of candidate paper IDs.
    """
    print("\nBuilding relevance score matrix...")
    print(f"This will make ~{len(claims) * CONFIG['TOP_K_CANDIDATES']} LLM calls.")
    print("Using advanced multi-section scoring with PRECISE granular values (0.00-1.00)...\n")
    
    llm_scorer = ChatOpenAI(
        model=CONFIG['LLM_MODEL'],
        temperature=CONFIG['LLM_TEMP_SCORING']
    )
    
    # Create paper lookup
    papers_by_id = {p['id']: p for p in papers}
    
    scores = defaultdict(dict)  # scores[claim_id][paper_id] = score
    candidates = {}  # candidates[claim_id] = [paper_ids]
    
    for claim in claims:
        claim_id = claim['id']
        print(f"\n[{claim_id}/{len(claims)}] Scoring papers for claim:")
        print(f"  '{claim['claim'][:80]}...'")
        
        # Get candidate papers via vector similarity
        candidate_ids = get_candidate_papers(
            claim['claim'],
            collection,
            top_k=CONFIG['TOP_K_CANDIDATES']
        )
        candidates[claim_id] = candidate_ids
        
        print(f"  Vector search returned {len(candidate_ids)} candidates")
        print(f"  Scoring with precise multi-section analysis...", end="", flush=True)
        
        # Score each candidate
        for paper_id in candidate_ids:
            paper = papers_by_id[paper_id]
            score = score_paper_for_claim(claim['claim'], paper, llm_scorer)
            scores[claim_id][paper_id] = score
            time.sleep(0.2)  # Slightly longer delay for more complex calls
        
        # Show top scores with precise values
        sorted_scores = sorted(scores[claim_id].items(), key=lambda x: x[1], reverse=True)
        print(f" Done.")
        print(f"  Top 3 scores: ", end="")
        for paper_id, score in sorted_scores[:3]:
            print(f"[{paper_id}]={score:.2f} ", end="")
        print()
    
    print(f"\n✓ Relevance matrix complete: {len(scores)} claims × {CONFIG['TOP_K_CANDIDATES']} papers")
    
    # Calculate score distribution statistics
    all_scores = [score for claim_scores in scores.values() for score in claim_scores.values()]
    if all_scores:
        print(f"\nScore distribution:")
        print(f"  Mean: {np.mean(all_scores):.3f}")
        print(f"  Median: {np.median(all_scores):.3f}")
        print(f"  Std Dev: {np.std(all_scores):.3f}")
        print(f"  Min: {min(all_scores):.2f}, Max: {max(all_scores):.2f}")
        print(f"  Scores > 0.70: {sum(1 for s in all_scores if s > 0.70)}/{len(all_scores)}")
        print(f"  Scores 0.40-0.70: {sum(1 for s in all_scores if 0.40 <= s <= 0.70)}/{len(all_scores)}")
        print(f"  Scores < 0.40: {sum(1 for s in all_scores if s < 0.40)}/{len(all_scores)}\n")
    
    return scores, candidates

# Build relevance scores
relevance_scores, candidate_papers = build_relevance_matrix(claims, papers, collection)


Building relevance score matrix...
This will make ~176 LLM calls.
Using advanced multi-section scoring with PRECISE granular values (0.00-1.00)...


[1/22] Scoring papers for claim:
  'Adaptive navigation of multirobot systems through physical environments has succ...'
  Vector search returned 8 candidates
  Scoring with precise multi-section analysis... Done.
  Top 3 scores: [17]=0.58 [8]=0.47 [27]=0.47 

[2/22] Scoring papers for claim:
  'Environments are modeled as scalar fields representing quantities like temperatu...'
  Vector search returned 8 candidates
  Scoring with precise multi-section analysis... Done.
  Top 3 scores: [17]=0.85 [3]=0.85 [47]=0.82 

[3/22] Scoring papers for claim:
  'Control primitives based on gradient methods enable robots to find extrema, sadd...'
  Vector search returned 8 candidates
  Scoring with precise multi-section analysis... Done.
  Top 3 scores: [18]=0.82 [17]=0.52 [21]=0.52 

[4/22] Scoring papers for claim:
  'Scalar field approaches have p

## Step 5: Optimize Citation Assignment with Simulated Annealing

In [35]:
def check_constraints(assignment, config):
    """
    Check if assignment satisfies all constraints.
    assignment: dict[claim_id] = [paper_ids]
    """
    # Count citations per paper
    paper_counts = Counter()
    for paper_list in assignment.values():
        paper_counts.update(paper_list)

    # Check max cites per paper
    if any(count > config['MAX_CITES_PER_PAPER'] for count in paper_counts.values()):
        return False
    
    # Check min papers used
    if len(paper_counts) < config['MIN_PAPERS']:
        return False
    
    # Check cites per claim
    for paper_list in assignment.values():
        num_cites = len(paper_list)
        if num_cites < config['CITES_PER_CLAIM_MIN'] or num_cites > config['CITES_PER_CLAIM_MAX']:
            return False
    
    return True

def calculate_objective(assignment, scores, config):
    """
    Calculate objective function value for an assignment.
    Higher is better. Combines:
    - Total relevance scores
    - Diversity bonus (more papers used)
    - Evenness bonus (balanced citation distribution)
    """
    if not check_constraints(assignment, config):
        return -1e9  # Invalid assignment
    
    # Total relevance
    total_relevance = 0.0
    for claim_id, paper_list in assignment.items():
        for paper_id in paper_list:
            total_relevance += scores[claim_id].get(paper_id, 0.0)
    
    # Diversity: count unique papers used
    paper_counts = Counter()
    for paper_list in assignment.values():
        paper_counts.update(paper_list)
    
    num_papers_used = len(paper_counts)
    diversity_bonus = num_papers_used * 0.5  # Encourage using more papers
    
    # Evenness: penalize high variance in paper usage
    if paper_counts:
        counts = list(paper_counts.values())
        mean_count = np.mean(counts)
        variance = np.var(counts)
        evenness_bonus = -variance * 0.1  # Penalize uneven distribution
    else:
        evenness_bonus = 0
    
    objective = total_relevance + diversity_bonus + evenness_bonus
    return objective

def generate_initial_assignment(claims, scores, candidates, config):
    """
    Generate initial feasible assignment using greedy approach.
    """
    assignment = {}
    paper_usage = Counter()
    
    for claim in claims:
        claim_id = claim['id']
        
        # Get candidate papers sorted by score
        candidates_list = [(pid, scores[claim_id].get(pid, 0.0)) 
                          for pid in candidates[claim_id]]
        candidates_list.sort(key=lambda x: x[1], reverse=True)
        
        # Skip if all scores too low
        if all(score < config['WEAK_MATCH_THRESHOLD'] for _, score in candidates_list):
            assignment[claim_id] = []
            continue
        
        # Select papers greedily
        selected = []
        for paper_id, score in candidates_list:
            if score < config['WEAK_MATCH_THRESHOLD']:
                break
            if paper_usage[paper_id] < config['MAX_CITES_PER_PAPER']:
                selected.append(paper_id)
                paper_usage[paper_id] += 1
                if len(selected) >= config['CITES_PER_CLAIM_MAX']:
                    break
        
        # Ensure minimum citations
        if len(selected) < config['CITES_PER_CLAIM_MIN']:
            for paper_id, score in candidates_list:
                if paper_id not in selected and paper_usage[paper_id] < config['MAX_CITES_PER_PAPER']:
                    selected.append(paper_id)
                    paper_usage[paper_id] += 1
                    if len(selected) >= config['CITES_PER_CLAIM_MIN']:
                        break
        
        assignment[claim_id] = selected
    
    # Ensure minimum papers used - add random papers to claims if needed
    while len(paper_usage) < config['MIN_PAPERS']:
        # Find a claim that can accept more citations
        for claim in claims:
            claim_id = claim['id']
            if len(assignment[claim_id]) < config['CITES_PER_CLAIM_MAX']:
                # Find unused paper with decent score
                for paper_id in candidates[claim_id]:
                    if paper_id not in paper_usage or paper_usage[paper_id] == 0:
                        if scores[claim_id].get(paper_id, 0.0) >= config['WEAK_MATCH_THRESHOLD']:
                            assignment[claim_id].append(paper_id)
                            paper_usage[paper_id] += 1
                            break
                if len(paper_usage) >= config['MIN_PAPERS']:
                    break
        else:
            break  # Can't add more
    
    return assignment

def simulated_annealing_step(current_assignment, scores, candidates, config, temperature):
    """
    Perform one step of simulated annealing.
    Returns new assignment and whether it was accepted.
    """
    # Copy current assignment
    new_assignment = {k: list(v) for k, v in current_assignment.items()}
    
    # Choose random modification
    modification = random.choice(['swap', 'replace', 'add', 'remove'])
    
    claim_ids = list(new_assignment.keys())
    claim_id = random.choice(claim_ids)
    
    if modification == 'swap' and len(new_assignment[claim_id]) >= 2:
        # Swap two papers within a claim
        idx1, idx2 = random.sample(range(len(new_assignment[claim_id])), 2)
        new_assignment[claim_id][idx1], new_assignment[claim_id][idx2] = \
            new_assignment[claim_id][idx2], new_assignment[claim_id][idx1]
    
    elif modification == 'replace' and new_assignment[claim_id]:
        # Replace one paper with another from candidates
        idx = random.randint(0, len(new_assignment[claim_id]) - 1)
        old_paper = new_assignment[claim_id][idx]
        available = [p for p in candidates[claim_id] if p not in new_assignment[claim_id]]
        if available:
            new_paper = random.choice(available)
            new_assignment[claim_id][idx] = new_paper
    
    elif modification == 'add' and len(new_assignment[claim_id]) < config['CITES_PER_CLAIM_MAX']:
        # Add a paper
        available = [p for p in candidates[claim_id] if p not in new_assignment[claim_id]]
        if available:
            new_paper = random.choice(available)
            new_assignment[claim_id].append(new_paper)
    
    elif modification == 'remove' and len(new_assignment[claim_id]) > config['CITES_PER_CLAIM_MIN']:
        # Remove a paper
        if new_assignment[claim_id]:
            idx = random.randint(0, len(new_assignment[claim_id]) - 1)
            new_assignment[claim_id].pop(idx)
    
    # Evaluate new assignment
    current_obj = calculate_objective(current_assignment, scores, config)
    new_obj = calculate_objective(new_assignment, scores, config)
    
    # Accept or reject
    if new_obj > current_obj:
        return new_assignment, True
    else:
        # Accept with probability based on temperature
        delta = new_obj - current_obj
        probability = np.exp(delta / temperature) if temperature > 0 else 0
        if random.random() < probability:
            return new_assignment, True
        else:
            return current_assignment, False

def run_simulated_annealing(claims, scores, candidates, config):
    """
    Run simulated annealing optimization multiple times and return best result.
    """
    print(f"\nRunning Simulated Annealing Optimization")
    print(f"  Runs: {config['SA_RUNS']}")
    print(f"  Iterations per run: {config['SA_ITERATIONS']}")
    print(f"  Total iterations: {config['SA_RUNS'] * config['SA_ITERATIONS']}\n")
    
    best_assignment = None
    best_objective = -float('inf')
    
    for run in range(config['SA_RUNS']):
        print(f"\nRun {run+1}/{config['SA_RUNS']}")
        
        # Generate initial assignment
        print("  Generating initial assignment...", end="", flush=True)
        current = generate_initial_assignment(claims, scores, candidates, config)
        current_obj = calculate_objective(current, scores, config)
        print(f" Objective: {current_obj:.2f}")
        
        if current_obj == -1e9:
            print("  Warning: Initial assignment violates constraints!")
            continue
        
        # Simulated annealing
        temperature = config['SA_INITIAL_TEMP']
        accepted = 0
        
        for iteration in range(config['SA_ITERATIONS']):
            current, was_accepted = simulated_annealing_step(
                current, scores, candidates, config, temperature
            )
            
            if was_accepted:
                accepted += 1
            
            temperature *= config['SA_COOLING_RATE']
            temperature = max(temperature, config['SA_MIN_TEMP'])
            
            # Progress report
            if (iteration + 1) % 50 == 0:
                current_obj = calculate_objective(current, scores, config)
                print(f"    Iteration {iteration+1}: Obj={current_obj:.2f}, "
                      f"Temp={temperature:.4f}, Accepted={accepted}/{iteration+1}")
        
        # Final objective
        final_obj = calculate_objective(current, scores, config)
        print(f"  Final objective: {final_obj:.2f}")
        
        # Update best
        if final_obj > best_objective:
            best_objective = final_obj
            best_assignment = current
            print(f"  ★ New best objective: {best_objective:.2f}")
    
    print(f"\n✓ Optimization complete!")
    print(f"  Best objective: {best_objective:.2f}\n")
    
    return best_assignment

# Run optimization
optimal_assignment = run_simulated_annealing(claims, relevance_scores, candidate_papers, CONFIG)


Running Simulated Annealing Optimization
  Runs: 100
  Iterations per run: 2200
  Total iterations: 220000


Run 1/100
  Generating initial assignment... Objective: 42.21
    Iteration 50: Obj=39.41, Temp=0.7742, Accepted=46/50
    Iteration 100: Obj=36.21, Temp=0.5449, Accepted=88/100
    Iteration 150: Obj=34.56, Temp=0.3835, Accepted=130/150
    Iteration 200: Obj=33.64, Temp=0.2699, Accepted=169/200
    Iteration 250: Obj=36.96, Temp=0.1900, Accepted=204/250
    Iteration 300: Obj=39.24, Temp=0.1337, Accepted=235/300
    Iteration 350: Obj=40.13, Temp=0.0941, Accepted=270/350
    Iteration 400: Obj=40.90, Temp=0.0662, Accepted=298/400
    Iteration 450: Obj=41.91, Temp=0.0466, Accepted=330/450
    Iteration 500: Obj=42.05, Temp=0.0328, Accepted=360/500
    Iteration 550: Obj=42.87, Temp=0.0231, Accepted=391/550
    Iteration 600: Obj=43.21, Temp=0.0163, Accepted=418/600
    Iteration 650: Obj=43.50, Temp=0.0114, Accepted=445/650
    Iteration 700: Obj=43.69, Temp=0.0081, Accepted=

In [45]:
# Analyze optimal assignment
print("\n" + "="*60)
print("OPTIMAL CITATION ASSIGNMENT ANALYSIS")
print("="*60)

# Count papers used
paper_counts = Counter()
for paper_list in optimal_assignment.values():
    paper_counts.update(paper_list)

print(f"\nTotal papers used: {len(paper_counts)}")
print(f"Total citations: {sum(paper_counts.values())}")
print(f"\nPaper usage distribution:")
for count, num_papers in Counter(paper_counts.values()).items():
    print(f"  {num_papers} papers cited {count} time(s)")

print(f"\nCitations per claim distribution:")
claim_citation_counts = Counter(len(papers) for papers in optimal_assignment.values())
for count, num_claims in sorted(claim_citation_counts.items()):
    print(f"  {num_claims} claims with {count} citation(s)")

# Total relevance score
total_relevance = 0.0
for claim_id, paper_list in optimal_assignment.items():
    for paper_id in paper_list:
        total_relevance += relevance_scores[claim_id].get(paper_id, 0.0)

print(f"\nTotal relevance score: {total_relevance:.2f}")
print(f"Average relevance per citation: {total_relevance/sum(paper_counts.values()):.3f}")


OPTIMAL CITATION ASSIGNMENT ANALYSIS

Total papers used: 41
Total citations: 66

Paper usage distribution:
  31 papers cited 1 time(s)
  2 papers cited 4 time(s)
  3 papers cited 5 time(s)
  2 papers cited 3 time(s)
  3 papers cited 2 time(s)

Citations per claim distribution:
  22 claims with 3 citation(s)

Total relevance score: 28.12
Average relevance per citation: 0.426


## Step 6: Generate Output Files

In [49]:
# Write output files
print("\n" + "="*60)
print("WRITING OUTPUT FILES")
print("="*60)

# Write introduction with citations
print(f"\nWriting {CONFIG['OUTPUT_INTRO']}...")
with open(CONFIG['OUTPUT_INTRO'], 'w', encoding='utf-8') as f:
    f.write("Introduction:\n\n")
    f.write(cited_introduction)
    f.write("\n")
print("✓ Introduction with citations saved")

# Write references
print(f"\nWriting {CONFIG['OUTPUT_REFS']}...")
with open(CONFIG['OUTPUT_REFS'], 'w', encoding='utf-8') as f:
    f.write("References\n")
    f.write("="*60 + "\n\n")
    for ref in references:
        f.write(ref + "\n\n")
print("✓ References saved")

print("\n" + "="*60)
print("CITATION PIPELINE COMPLETE!")
print("="*60)
print(f"\n📄 Output files:")
print(f"   - {CONFIG['OUTPUT_INTRO']}")
print(f"   - {CONFIG['OUTPUT_REFS']}")
print(f"\n📊 Statistics:")
print(f"   - {len(claims)} claims identified")
print(f"   - {len(paper_to_citation)} papers cited")
print(f"   - {sum(len(cites) for cites in claim_citations.values())} total citations")
print("\n✓ Ready for review!\n")


WRITING OUTPUT FILES

Writing ./introduction_with_citations.txt...


NameError: name 'cited_introduction' is not defined

In [ ]:
# Preview outputs
print("\n" + "="*60)
print("PREVIEW: INTRODUCTION WITH CITATIONS (first 800 chars)")
print("="*60)
print(cited_introduction[:800] + "...\n")

print("="*60)
print("PREVIEW: REFERENCES (first 10)")
print("="*60)
for ref in references[:10]:
    print(ref)


PREVIEW: INTRODUCTION WITH CITATIONS (first 800 chars)
Introduction:

Multirobot systems navigating through physical vector fields face a fundamental challenge when operating near critical points. [1-4] Consider a team of underwater robots monitoring an oceanic vortex or atmospheric drones tracking a developing cyclone. These robots must maintain safe distances from dangerous field centers while gathering crucial data. Yet without knowing where these centers actually lie, only their general direction, the robots risk being pulled into dangerous zones or ejected from their monitoring positions. This problem becomes even more complex when the field evolves during the mission, with vortices strengthening, weakening, or drifting across the environment.
Vector fields present unique navigation challenges that distinguish them from the extensively st...

PREVIEW: REFERENCES (first 10)
[1] Proceedings of the ASME 2019 International Design Engineering Technical Conferences and Computers an - p

In [41]:
relevance_scores

defaultdict(dict,
            {1: {31: 0.1,
              39: 0.0,
              8: 0.47,
              17: 0.58,
              53: 0.27,
              47: 0.28,
              36: 0.38,
              27: 0.47},
             2: {38: 0.72,
              17: 0.85,
              19: 0.58,
              47: 0.82,
              8: 0.29,
              5: 0.62,
              25: 0.72,
              3: 0.85},
             3: {25: 0.34,
              17: 0.52,
              21: 0.52,
              27: 0.12,
              19: 0.05,
              18: 0.82,
              8: 0.19,
              35: 0.05},
             4: {17: 0.85,
              47: 0.62,
              8: 0.12,
              25: 0.82,
              19: 0.67,
              2: 0.66,
              5: 0.62,
              11: 0.05},
             5: {38: 0.29,
              44: 0.47,
              17: 0.1,
              47: 0.48,
              3: 0.1,
              23: 0.33,
              5: 0.12,
              8: 0.18},
             6: {

In [46]:
  # ============================================================================
  # TOP 30 MOST RELEVANT PAPERS - Run this after the notebook
  # ============================================================================

  import numpy as np
  from collections import defaultdict

  print("="*70)
  print("TOP 50 MOST RELEVANT PAPERS")
  print("="*70)

  # Calculate aggregate relevance score for each paper
  paper_relevance = defaultdict(lambda: {'total_score': 0.0, 'max_score': 0.0, 'num_claims': 0, 'scores': []})

  for claim_id, paper_scores in relevance_scores.items():
      for paper_id, score in paper_scores.items():
          paper_relevance[paper_id]['total_score'] += score
          paper_relevance[paper_id]['max_score'] = max(paper_relevance[paper_id]['max_score'], score)
          paper_relevance[paper_id]['num_claims'] += 1
          paper_relevance[paper_id]['scores'].append(score)

  # Calculate average score for each paper
  for paper_id in paper_relevance:
      scores = paper_relevance[paper_id]['scores']
      paper_relevance[paper_id]['avg_score'] = np.mean(scores)

  # Create papers lookup
  papers_by_id = {p['id']: p for p in papers}

  # Sort by total relevance score
  ranked_papers = sorted(
      paper_relevance.items(),
      key=lambda x: x[1]['total_score'],
      reverse=True
  )

  print(f"\nRanked by TOTAL relevance across all claims:\n")
  print(f"{'Rank':<5} {'ID':<4} {'Total':<7} {'Max':<6} {'Avg':<6} {'#Claims':<8} {'Cited':<6} Title")
  print("-"*110)

  for rank, (paper_id, stats) in enumerate(ranked_papers[:50], 1):
      paper = papers_by_id[paper_id]
      is_cited = paper_id in paper_to_citation
      cite_marker = "✓" if is_cited else " "
      cite_num = f"[{paper_to_citation[paper_id]}]" if is_cited else ""
      title = paper['title'][:60]
      print(f"{rank:<5} {paper_id:<4} {stats['total_score']:>6.2f}  {stats['max_score']:>5.2f}  {stats['avg_score']:>5.2f}  {stats['num_claims']:<8} {cite_marker:<3}{cite_num:<4} {title}")

  # Alternative ranking by MAX score
  print(f"\n" + "="*70)
  print("ALTERNATIVE: Ranked by MAX relevance (best single claim match):\n")

  ranked_by_max = sorted(
      paper_relevance.items(),
      key=lambda x: x[1]['max_score'],
      reverse=True
  )

  print(f"{'Rank':<5} {'ID':<4} {'Max':<6} {'Total':<7} {'Avg':<6} {'Cited':<6} Title")
  print("-"*110)

  for rank, (paper_id, stats) in enumerate(ranked_by_max[:50], 1):
      paper = papers_by_id[paper_id]
      is_cited = paper_id in paper_to_citation
      cite_marker = "✓" if is_cited else " "
      cite_num = f"[{paper_to_citation[paper_id]}]" if is_cited else ""
      title = paper['title'][:65]
      print(f"{rank:<5} {paper_id:<4} {stats['max_score']:>5.2f}  {stats['total_score']:>6.2f}  {stats['avg_score']:>5.2f}  {cite_marker:<3}{cite_num:<4} {title}")

  # Show papers that scored high but weren't cited
  print(f"\n" + "="*70)
  print("HIGH-SCORING PAPERS NOT CITED (potential additions):\n")

  uncited_high_scorers = [
      (paper_id, stats) for paper_id, stats in ranked_papers
      if paper_id not in paper_to_citation and stats['total_score'] > 0.5
  ]

  if uncited_high_scorers:
      print(f"{'ID':<4} {'Total':<7} {'Max':<6} {'Avg':<6} Title")
      print("-"*90)
      for paper_id, stats in uncited_high_scorers[:35]:
          paper = papers_by_id[paper_id]
          title = paper['title'][:70]
          print(f"{paper_id:<4} {stats['total_score']:>6.2f}  {stats['max_score']:>5.2f}  {stats['avg_score']:>5.2f}  {title}")
  else:
      print("None - all high-scoring papers were cited!")

  print("\n" + "="*70)

TOP 50 MOST RELEVANT PAPERS

Ranked by TOTAL relevance across all claims:

Rank  ID   Total   Max    Avg    #Claims  Cited  Title
--------------------------------------------------------------------------------------------------------------


NameError: name 'paper_to_citation' is not defined

In [ ]:
# ============================================================================
# CLAIMS WITH TOP 5 SUPPORTING PAPERS - Run this after the notebook
# ============================================================================

print("="*70)
print("CLAIMS WITH TOP 5 SUPPORTING PAPERS")
print("="*70)

# Create papers lookup
papers_by_id = {p['id']: p for p in papers}

# Sort claims by position (order in introduction)
sorted_claims = sorted(claims, key=lambda c: c['position'])

for claim in sorted_claims:
    claim_id = claim['id']
    claim_text = claim['claim']

    # Get all paper scores for this claim
    paper_scores = relevance_scores.get(claim_id, {})

    # Sort by score descending
    ranked_papers = sorted(paper_scores.items(), key=lambda x: x[1], reverse=True)

    # Get citation numbers for this claim
    citation_nums = claim_citations.get(claim_id, [])
    citation_str = f"[{','.join(map(str, citation_nums))}]" if citation_nums else "[NO CITATIONS]"

    print(f"\n{'='*70}")
    print(f"CLAIM {claim_id} {citation_str}")
    print(f"Topic: {claim['topic']}")
    print(f"{'='*70}")
    print(f"\"{claim_text}\"")
    print(f"\nTop 5 Supporting Papers:")
    print(f"{'Rank':<5} {'Score':<7} {'Cited':<6} {'Paper ID':<9} Title")
    print("-"*70)

    for rank, (paper_id, score) in enumerate(ranked_papers[:5], 1):
        paper = papers_by_id.get(paper_id)
        if paper:
            # Check if this specific paper is cited for this claim
            is_cited_here = paper_id in optimal_assignment.get(claim_id, [])

            # Check if paper is cited anywhere
            cite_num = paper_to_citation.get(paper_id)

            if is_cited_here:
                cite_marker = f"✓[{cite_num}]"
            elif cite_num:
                cite_marker = f" [{cite_num}]"
            else:
                cite_marker = ""

            print(f"{rank:<5} {score:>5.2f}   {cite_marker:<6} {paper_id:<9} {paper['title'][:50]}")

    # Show which papers were actually selected
    selected_papers = optimal_assignment.get(claim_id, [])
    if selected_papers:
        print(f"\n  ✓ Selected papers for this claim: {selected_papers}")
        # Show if any selected papers weren't in top 5
        top_5_ids = [pid for pid, _ in ranked_papers[:5]]
        not_in_top5 = [pid for pid in selected_papers if pid not in top_5_ids]
        if not_in_top5:
            print(f"  ⚠ Note: Papers {not_in_top5} were selected but not in top 5 by score")
            print(f"      (likely chosen for diversity/constraint satisfaction)")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

# Calculate statistics
total_claims = len(claims)
claims_with_top_match = 0
claims_citing_top5 = 0

for claim in claims:
    claim_id = claim['id']
    paper_scores = relevance_scores.get(claim_id, {})
    ranked_papers = sorted(paper_scores.items(), key=lambda x: x[1], reverse=True)
    selected_papers = optimal_assignment.get(claim_id, [])

    if ranked_papers and selected_papers:
        top_paper_id = ranked_papers[0][0]
        if top_paper_id in selected_papers:
            claims_with_top_match += 1

        top_5_ids = [pid for pid, _ in ranked_papers[:5]]
        if any(pid in top_5_ids for pid in selected_papers):
            claims_citing_top5 += 1

print(f"\nCitation Quality Metrics:")
print(f"  Claims citing their #1 ranked paper: {claims_with_top_match}/{total_claims} ({100*claims_with_top_match/total_claims:.1f}%)")
print(f"  Claims citing at least one top-5 paper: {claims_citing_top5}/{total_claims} ({100*claims_citing_top5/total_claims:.1f}%)")
print(f"\n  (Lower percentages are OK - optimizer balances quality with diversity/constraints)")

print("="*70)

CLAIMS WITH TOP 5 SUPPORTING PAPERS

CLAIM 1 [1,2,3,4]
Topic: navigation challenges
"Multirobot systems navigating through physical vector fields face a fundamental challenge when operating near critical points"

Top 5 Supporting Papers:
Rank  Score   Cited  Paper ID  Title
----------------------------------------------------------------------
1      0.57   ✓[3]   4         Motion Planning and Collision Avoidance using Navi
2      0.47   ✓[4]   18        Initial Study of Multirobot Adaptive Navigation fo
3      0.47   ✓[1]   21        1 © 2019 by ASME
4      0.33   ✓[2]   5         Vector Field-based Collision Avoidance for
5      0.33    [9]   53        Obstacle Avoidance Using Complex Vector Fields

  ✓ Selected papers for this claim: [21, 5, 4, 18]

CLAIM 3 [5,6,7,8]
Topic: scalar field navigation
"Scalar fields like temperature or chemical concentration can be navigated using well established gradient methods"

Top 5 Supporting Papers:
Rank  Score   Cited  Paper ID  Title
---------

In [47]:
  # ============================================================================
  # CITATIONS TO REVIEW (lowest scores first)
  # ============================================================================

  print("="*70)
  print("REVIEW THESE CITATIONS (lowest relevance scores)")
  print("="*70)

  review_list = []

  for claim_id, paper_ids in optimal_assignment.items():
      claim = next(c for c in claims if c['id'] == claim_id)

      for paper_id in paper_ids:
          score = relevance_scores[claim_id].get(paper_id, 0.0)
          paper = papers_by_id[paper_id]
          cite_num = paper_to_citation[paper_id]

          review_list.append({
              'claim_id': claim_id,
              'paper_id': paper_id,
              'cite_num': cite_num,
              'score': score,
              'claim_text': claim['claim'][:80],
              'paper_title': paper['title'][:60]
          })

  # Sort by score (lowest first - these need the most scrutiny)
  review_list.sort(key=lambda x: x['score'])

  print("\nCitations with score < 0.40 (consider removing):\n")
  print(f"{'Score':<7} {'Cite':<6} {'Claim':<6} Paper Title")
  print("-"*90)

  low_score_count = 0
  for item in review_list:
      if item['score'] < 0.30:
          low_score_count += 1
          print(f"{item['score']:>5.2f}   [{item['cite_num']:<3}]  Claim {item['claim_id']:<3} {item['paper_title']}")

  print(f"\n  Total citations with score < 0.40: {low_score_count}")

  print("\nCitations with score 0.40-0.60 (verify these carefully):\n")
  print(f"{'Score':<7} {'Cite':<6} {'Claim':<6} Paper Title")
  print("-"*90)

  medium_score_count = 0
  for item in review_list:
      if 0.30 <= item['score'] < 0.60:
          medium_score_count += 1
          print(f"{item['score']:>5.2f}   [{item['cite_num']:<3}]  Claim {item['claim_id']:<3} {item['paper_title']}")

  print(f"\n  Total citations with score 0.40-0.60: {medium_score_count}")
  print(f"\nCitations with score ≥ 0.60: {len(review_list) - low_score_count - medium_score_count} (probably fine)")

  print("\n" + "="*70)

REVIEW THESE CITATIONS (lowest relevance scores)


NameError: name 'paper_to_citation' is not defined